In [ ]:
#!/usr/bin/env python3
"""
Lane Count Extraction Script

This script reads a CSV file listing video frames and their GPS coordinates (latitude, longitude),
queries the OpenStreetMap Overpass API for the number of lanes at each location, and writes out a 
new CSV with an added 'lane_count' column.

Input CSV format (from Data_Gen.ipynb):
    video_name, frame_number, lat, lon

Output CSV format:
    video_name, frame_number, lat, lon, lane_count

For each coordinate, the script finds the nearest road in OSM (within a small radius) and retrieves 
its 'lanes' tag. If a direct 'lanes' tag is missing but 'lanes:forward'/'lanes:backward' are present, 
those are summed to estimate total lanes:contentReference[oaicite:15]{index=15}. If no lane info is available, lane_count is set to "N/A".

Overpass Query:
We use Overpass QL's around clause to find highways near the coordinate:contentReference[oaicite:16]{index=16}. The query looks like:
    way(around:15, LAT, LON)["highway"];
    out tags;
This returns highway ways within 15m of the point, with their tags (id and tags only):contentReference[oaicite:17]{index=17}.

Rate Limiting & Retries:
To avoid hitting Overpass rate limits, the script pauses briefly between requests and retries failed queries up to 3 times.
"""
import csv
import requests
import time

# Overpass API endpoint
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# Overpass query template for finding highways around a coordinate
QUERY_TEMPLATE = '[out:json]; way(around:{radius},{lat},{lon})[highway]; out tags;'

def get_lane_count_for_location(latitude, longitude, radius=15, max_retries=3):
    """
    Query Overpass API for lane count at the given latitude and longitude.
    Returns an integer number of lanes if found, or None if no data.
    """
    # Format the Overpass QL query
    query = QUERY_TEMPLATE.format(radius=radius, lat=latitude, lon=longitude)
    headers = {"User-Agent": "LaneCountScript/1.0"}  # identify our tool to Overpass
    for attempt in range(max_retries):
        try:
            response = requests.get(OVERPASS_URL, params={'data': query}, headers=headers, timeout=10)
        except requests.RequestException as e:
            # Network or connection error
            wait = 2 * (attempt + 1)
            time.sleep(wait)
            continue
        if response.status_code != 200:
            # Server returned an error (e.g. 429 Too Many Requests)
            wait = 2 * (attempt + 1)
            time.sleep(wait)
            continue
        # If we got a 200 OK response, attempt to parse JSON
        try:
            data = response.json()
        except ValueError:
            # Invalid JSON (shouldn't happen with out:json)
            return None
        # Look through returned ways for lane information
        lane_count = None
        for element in data.get('elements', []):
            tags = element.get('tags', {})
            if 'lanes' in tags:
                # Use the first way that has an explicit 'lanes' tag
                try:
                    lane_count = int(tags['lanes'])
                except ValueError:
                    # If lanes tag is non-numeric, skip it
                    lane_count = None
                if lane_count is not None:
                    break
        if lane_count is None:
            # If no direct 'lanes' tag found, check for lanes:forward/backward
            for element in data.get('elements', []):
                tags = element.get('tags', {})
                if 'lanes:forward' in tags or 'lanes:backward' in tags:
                    # Sum forward/backward lanes if present
                    try:
                        fwd = int(tags.get('lanes:forward', 0))
                        back = int(tags.get('lanes:backward', 0))
                        total = fwd + back
                        # Only consider it a valid count if either forward or backward was present
                        if fwd or back:
                            lane_count = total
                            break
                    except ValueError:
                        continue
        return lane_count  # could be None if still not found
    # If we reached max_retries without success, return None
    return None

def add_lane_counts_to_csv(input_csv_path, output_csv_path):
    """
    Read input CSV, fetch lane counts for each coordinate, and write output CSV with lane_count column.
    """
    with open(input_csv_path, newline='') as infile, open(output_csv_path, 'w', newline='') as outfile:
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames + ['lane_count'] if reader.fieldnames else None
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        if fieldnames:
            writer.writeheader()
        count_total = 0
        count_found = 0
        for row in reader:
            count_total += 1
            lat = row.get('lat') or row.get('latitude')
            lon = row.get('lon') or row.get('longitude')
            if lat is None or lon is None:
                row['lane_count'] = "N/A"
                writer.writerow(row)
                continue
            try:
                # Convert coordinates to float
                lat_val = float(lat)
                lon_val = float(lon)
            except ValueError:
                row['lane_count'] = "N/A"
                writer.writerow(row)
                continue
            # Query Overpass for lane count at this coordinate
            lane_count = get_lane_count_for_location(lat_val, lon_val)
            if lane_count is None:
                row['lane_count'] = "N/A"
            else:
                row['lane_count'] = str(lane_count)
                count_found += 1
            writer.writerow(row)
            # Throttle requests: brief pause to respect API limits
            time.sleep(1)  # 1 second delay between queries
    print(f"[done] Processed {count_total} locations, found lane data for {count_found} of them.")
    print(f"Output saved to {output_csv_path}")

In [ ]:
input_path =r"C:\Users\HP\Documents\base\location\20250826_32351PMByGPSMapCamera\20250826_32351PMByGPSMapCamera.csv"
output_path = r"C:\Users\HP\Documents\base\location\20250826_32351PMByGPSMapCamera\20250826_32351PMByGPSMapCamera_Result.csv"

add_lane_counts_to_csv(input_path, output_path)